# Med-Guard — Training and Evaluation Pipeline

Signed-graph-based embedding propagation model that classifies user requests as **safe / unsafe** and predicts intent.

Heavy/reusable logic (config, data processing, graph construction, propagation, model, loss functions, training and inference) is kept in modules under `../src/`. This notebook calls them and analyzes the results.

## 1. Setup and Libraries

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import classification_report, confusion_matrix

from src.config import Config, IDX_TO_INTENT, INTENT_CLASS_WEIGHTS
from src import data, graph, propagation, evaluate
from src.model import SignedGraphSafetyModel
from src.train import train_model

## 2. Configuration

Data path, embedding model, graph/propagation thresholds, and training hyperparameters.

In [ ]:
cfg = Config()

In [ ]:
# Propagation convention and effective parameters (same as report Equation 31-32)
c_eff = cfg.NEG_LAMBDA * cfg.PROP_ALPHA_NEG
print(f"PROP_ALPHA_POS (TELEPORT / self-representation weight) : {cfg.PROP_ALPHA_POS}  -> neighbour weight {1 - cfg.PROP_ALPHA_POS:.2f}")
print(f"PROP_K (propagation steps)                              : {cfg.PROP_K}")
print(f"Negative delta effective coefficient c                  : {c_eff:.3f}   (NEG_LAMBDA x PROP_ALPHA_NEG)")
print(f"INDUCTIVE_SIM_THR                                       : {cfg.INDUCTIVE_SIM_THR}")

## 3. Data Loading and Embedding Extraction

Reading the CSV, applying the label/intent maps, and vectorizing the texts with a multilingual sentence-transformer.

In [ ]:
df            = data.load_dataset(cfg.DATA_PATH)
texts         = df["request"].tolist()
labels        = df["label"].values
intent_labels = df["intent_idx"].values

print("Dataset size:", len(df))

embeddings, encoder = data.embed_texts(texts, cfg.EMBED_MODEL)
print("Embeddings shape:", embeddings.shape)

## 4. Train / Calibration / Test Split

Embeddings are first split into train/test, then the train set is split into train/calibration, stratified by intent label; moved to tensors and device (GPU/CPU).

In [ ]:
split = data.split_train_cal_test(
    embeddings, labels, intent_labels, texts,
    test_size=cfg.TEST_SIZE,
    cal_size_within_train=cfg.CAL_SIZE_WITHIN_TRAIN,
    random_state=cfg.RANDOM_STATE)

print("Train size:", len(split["X_train"]))
print("Cal size  :", len(split["X_cal"]))
print("Test size :", len(split["X_test"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tensors = data.to_tensors(split, device)

## 5. Signed Graph Construction

Graph built on the training set based on k-NN similarity: **positive** (pull) edges between similar examples with the same label, **negative** (push) edges between similar examples with opposite labels on predefined confusable intent pairs.

In [ ]:
confusion_pairs = graph.build_confusion_pairs()

pos_i, pos_j, neg_u, neg_s, A_pos_t, A_neg_t = graph.build_signed_graph_separate(
    split["X_train"], split["y_train"], split["int_train"],
    confusion_pairs, device,
    k_graph=cfg.K_GRAPH,
    pos_sim_threshold=cfg.POS_SIM_THRESHOLD,
    neg_sim_threshold=cfg.NEG_SIM_THRESHOLD)

print("Positive edges:", len(pos_i))
print("Negative edges:", len(neg_u))

## 6. Inductive Neighbourhood Computation

Propagation functions are defined in `src.propagation` (transductive and inductive). Here, the nearest neighbourhoods in the training set are computed for unseen test/calibration examples.

In [ ]:
print("Computing inductive neighbourhood (test)...")
test_nbr_idx, test_nbr_wts = propagation.build_inductive_neighbourhood(
    split["X_test"], split["X_train"], k=cfg.K_INDUCTIVE, sim_threshold=cfg.INDUCTIVE_SIM_THR)

print("Computing inductive neighbourhood (cal)...")
cal_nbr_idx, cal_nbr_wts = propagation.build_inductive_neighbourhood(
    split["X_cal"], split["X_train"], k=cfg.K_INDUCTIVE, sim_threshold=cfg.INDUCTIVE_SIM_THR)

### Reproducibility

Model initialization and dropout are random. If the seed is not fixed, the same notebook gives slightly different results on each run. Run this cell **immediately before** the model.

In [ ]:
import random

random.seed(cfg.RANDOM_STATE)
np.random.seed(cfg.RANDOM_STATE)
torch.manual_seed(cfg.RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.RANDOM_STATE)
print("Seed:", cfg.RANDOM_STATE)

## 7. Model Architecture

A network branching from a shared trunk (`Linear → ReLU → Dropout`) into safety (binary) and intent (9-class) heads.

In [ ]:
model = SignedGraphSafetyModel(
    in_dim=split["X_train"].shape[1], num_intents=9).to(device)

## 8. Loss Functions and Optimizer

Weighted BCE (safety), class-weighted CE (intent), pull-loss, margin-based push-loss, and propagation-consistency loss.

In [ ]:
intent_weights_t = torch.tensor(INTENT_CLASS_WEIGHTS, device=device)
criterion_intent = nn.CrossEntropyLoss(weight=intent_weights_t)

optimizer = torch.optim.Adam(
    model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

## 9. Training (Training Loop)

In [ ]:
model = train_model(
    model, optimizer, criterion_intent,
    tensors["X_train_t"], tensors["y_train_t"], tensors["int_train_t"],
    A_pos_t, A_neg_t, pos_i, pos_j, neg_u, neg_s,
    cfg, device)

## 10. Inference — Inductive Propagation

Propagation features are computed on the training set using the trained model; for the calibration and test sets, these features are propagated through the inductive neighbourhood to produce predictions.

In [ ]:
print("Inference — separated inductive propagation...")

results = evaluate.run_inference(
    model, tensors["X_train_t"], tensors["X_cal_t"], tensors["X_test_t"],
    A_pos_t, A_neg_t,
    cal_nbr_idx, cal_nbr_wts,
    test_nbr_idx, test_nbr_wts,
    cfg, device)

## 11. Evaluation — Propagation Effect Analysis

Comparison of local (pre-propagation) and post-propagation predictions; examination of false negatives (FN) recovered by propagation.

In [ ]:
y_train    = split["y_train"]
y_test     = split["y_test"]
int_test   = split["int_test"]
texts_test = split["texts_test"]

probs_train_local = results["probs_train_local"]
probs_train_prop  = results["probs_train_prop"]
probs_test_local  = results["probs_test_local"]
probs_test        = results["probs_test"]

print("="*70)
print("SEPARATED EMBEDDING PROPAGATION ANALYSIS")
print("="*70)

print(f"\nTrain set:")
print(f"  Local  — unsafe avg: {probs_train_local[y_train==1].mean():.4f} | "
      f"safe avg: {probs_train_local[y_train==0].mean():.4f}")
print(f"  Prop   — unsafe avg: {probs_train_prop[y_train==1].mean():.4f} | "
      f"safe avg: {probs_train_prop[y_train==0].mean():.4f}")
print(f"  → Diff unsafe: {probs_train_prop[y_train==1].mean() - probs_train_local[y_train==1].mean():+.4f}")
print(f"  → Diff safe:   {probs_train_prop[y_train==0].mean() - probs_train_local[y_train==0].mean():+.4f}")

print(f"\nTest set (inductive):")
print(f"  Local  — unsafe avg: {probs_test_local[y_test==1].mean():.4f} | "
      f"safe avg: {probs_test_local[y_test==0].mean():.4f}")
print(f"  Prop   — unsafe avg: {probs_test[y_test==1].mean():.4f} | "
      f"safe avg: {probs_test[y_test==0].mean():.4f}")
print(f"  → Diff unsafe: {probs_test[y_test==1].mean() - probs_test_local[y_test==1].mean():+.4f}")
print(f"  → Diff safe:   {probs_test[y_test==0].mean() - probs_test_local[y_test==0].mean():+.4f}")


disagree = np.abs(probs_test_local - probs_test)
recovered = ((probs_test_local < 0.5) & (probs_test > 0.5) & (y_test == 1))
print(f"\nFN recovered by propagation: {recovered.sum()}")
if recovered.sum() > 0:
    for idx in np.where(recovered)[0]:
        print(f"  local={probs_test_local[idx]:.3f} → prop={probs_test[idx]:.3f} | "
              f"intent={IDX_TO_INTENT[int_test[idx]]} | {texts_test[idx]}")

print(f"\nAverage disagreement: {disagree.mean():.4f} | max: {disagree.max():.4f}")
print(f"High disagreement (>0.10) ratio: {(disagree > 0.10).mean():.4f}")

## 12. Evaluation — Classification Metrics

`classification_report` and confusion matrix on the test set with the final (post-propagation) predictions.

In [ ]:
y_pred_test_binary = (probs_test >= 0.5).astype(int)
y_pred_local       = (probs_test_local >= 0.5).astype(int)

print("="*70)
print(classification_report(y_test, y_pred_test_binary,
                             target_names=["safe","unsafe"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test_binary))

### 12b. Side-by-side comparison of local and propagated predictions

"Local" = the same trained model, without inductive propagation applied (the propagation loss during training still affected the model). **Single seed and 508 examples**: 1 example ≈ 0.2 points of accuracy, so don't over-interpret small differences. Use the multi-seed experiments in section 15 for a reliable comparison.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

print(f"{'':<26}{'acc':>8}{'safe rec':>10}{'unsafe rec':>12}{'AUROC':>8}")
for name, p in [("lokal (propagasyonsuz)", probs_test_local), ("propagasyonlu", probs_test)]:
    yp = (p >= 0.5).astype(int)
    print(f"{name:<26}{accuracy_score(y_test, yp):>8.4f}"
          f"{recall_score(y_test, yp, pos_label=0):>10.4f}"
          f"{recall_score(y_test, yp, pos_label=1):>12.4f}"
          f"{roc_auc_score(y_test, p):>8.4f}")

## 13. False Negative Analysis

Detailed list of examples the model failed to detect as "unsafe" (from lowest to highest probability).

In [ ]:
pred_int_test = results["pred_int_test"]

false_negatives = []
for i, text in enumerate(texts_test):
    if y_test[i] == 1 and y_pred_test_binary[i] == 0:
        false_negatives.append({
            "text":        text,
            "local":       float(probs_test_local[i]),
            "prop":        float(probs_test[i]),
            "true_intent": IDX_TO_INTENT[int_test[i]],
            "pred_intent": IDX_TO_INTENT[pred_int_test[i]]
        })

print("="*70)
print(f"BINARY FALSE NEGATIVES | Count = {len(false_negatives)}")
print("="*70)
for x in sorted(false_negatives, key=lambda z: z["prop"])[:30]:
    print("-"*90)
    print(f"LOCAL={x['local']:.4f} → PROP={x['prop']:.4f} | "
          f"TRUE={x['true_intent']} | PRED={x['pred_intent']}")
    print("TEXT:", x["text"])

## 14. Measuring the Direction-Preservation Condition

The proven statement is a **conditional** proposition: for $c=\lambda_{neg}\alpha_{neg}$, $m_i=\|H_{pos}[i]\|$ and $M=\max_j\|H_{pos}[j]\|$,

$$m_i > \frac{c}{1+c}\,M \;\Longrightarrow\; \langle H_{pos}[i],H_{final}[i]\rangle > 0 .$$

The condition is sufficient, not necessary. This cell measures what fraction of the training nodes satisfy the condition, and what the actual inner product/cosine is. For nodes with no negative neighbours, $H_{final}=(1+c)H_{pos}$, i.e. only scaling is applied, not push; these nodes are reported separately.

In [ ]:
model.eval()
with torch.no_grad():
    H_local, _, _ = model(tensors["X_train_t"])
    H_pos = propagation.signed_embedding_propagation_separate(
        H_local, A_pos_t, A_neg_t,
        alpha_pos=cfg.PROP_ALPHA_POS, alpha_neg=cfg.PROP_ALPHA_NEG,
        K=cfg.PROP_K, neg_lambda=0.0)                                  # pure positive representation
    delta = cfg.PROP_ALPHA_NEG * (torch.sparse.mm(A_neg_t, H_pos) - H_pos)
    H_fin = H_pos - cfg.NEG_LAMBDA * delta

    norms = torch.linalg.norm(H_pos, dim=1)
    M = norms.max().item()
    c = cfg.NEG_LAMBDA * cfg.PROP_ALPHA_NEG
    thr = c / (1 + c)

    cond      = norms > thr * M                                        # sufficient condition
    dots      = (H_pos * H_fin).sum(1)
    cosine    = dots / (norms * torch.linalg.norm(H_fin, dim=1) + 1e-12)
    has_neg   = torch.sparse.sum(A_neg_t, dim=1).to_dense() > 0        # is the A_neg row non-empty

print(f"c = {c:.3f}  ->  threshold c/(1+c) = {thr:.4f}")
print(f"min/max norm ratio          : {norms.min().item() / M:.4f}")
print(f"satisfies sufficient cond.  : {cond.float().mean().item():.4f}  (fraction of nodes)")
print(f"inner product > 0           : {(dots > 0).float().mean().item():.4f}")
print(f"nodes with negative nbr     : {has_neg.float().mean().item():.4f}")
if has_neg.any():
    print(f"  min cosine among these    : {cosine[has_neg].min().item():.4f}   avg: {cosine[has_neg].mean().item():.4f}")
print(f"nodes without negative nbr  : {(~has_neg).float().mean().item():.4f}  -> H_final = (1+c) H_pos (scaling only)")

## 15. Multi-Seed Experiments (`src/experiments.py`)

This notebook runs with a single seed and a single split. Since the effect of propagation is small, base the decision on multi-seed results rather than a single run. The experiments build the split, graph, and neighbourhoods once per seed and train all configurations on the same split; results are reported as a **paired** difference against the baseline. `--quick` uses 2 seeds; for a full run use `--seeds 5`.

Experiments: `ablation`, `alpha`, `k`, `neg`, `noise`, `label_eff` (and `all`). Outputs are written as CSV under `../results/`.

In [ ]:
from src import experiments

# Quick look (2 seeds). For a full run: "--seeds", "5" instead of "--quick"
experiments.main(["ablation", "--quick",
                  "--data-path", cfg.DATA_PATH,
                  "--embed-cache", "../data/embeddings_cache.npz",
                  "--out-dir", "../results"])

# Other experiments (each runs separately):
# experiments.main(["alpha",     "--quick", "--data-path", cfg.DATA_PATH, "--embed-cache", "../data/embeddings_cache.npz", "--out-dir", "../results"])
# experiments.main(["noise",     "--quick", "--data-path", cfg.DATA_PATH, "--embed-cache", "../data/embeddings_cache.npz", "--out-dir", "../results"])
# experiments.main(["label_eff", "--quick", "--data-path", cfg.DATA_PATH, "--embed-cache", "../data/embeddings_cache.npz", "--out-dir", "../results"])